# As estatísticas da aula 2 em SQL

## Task 2: traduzir as estatísticas da aula 2 para SQL e guardar a consulta no projeto

As estatísticas da aula 2 em SQL, com resultado **idêntico** ao da DataFrame API.

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-estatisticas-sql")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: o código dos vídeos 2.4 e 3.1, na ordem, com as cinco views do projeto.

In [2]:
# vídeo 2.4: escalações e Copas
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
        F.col("MatchesPlayed").alias("partidas_oficiais"),
        F.col("GoalsScored").alias("gols_oficiais"),
    )
)

# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")
escalacoes.createOrReplaceTempView("escalacoes")
copas.createOrReplaceTempView("copas")

## Gols por Copa em SQL

In [3]:
spark.sql("""
    SELECT ano,
           COUNT(*)                   AS partidas,
           SUM(total_gols)            AS gols,
           ROUND(AVG(total_gols), 2)  AS media_gols,
           ROUND(AVG(publico), 0)     AS publico_medio
    FROM partidas
    GROUP BY ano
    ORDER BY ano
""").show(5)

+----+--------+----+----------+-------------+
| ano|partidas|gols|media_gols|publico_medio|
+----+--------+----+----------+-------------+
|1930|      18|  70|      3.89|      32808.0|
|1934|      17|  70|      4.12|      21353.0|
|1938|      18|  84|      4.67|      20872.0|
|1950|      22|  88|       4.0|      47511.0|
|1954|      26| 140|      5.38|      29562.0|
+----+--------+----+----------+-------------+
only showing top 5 rows


## A tabela por seleção: CTE e UNION ALL

- **CTE** (`WITH nome AS (...)`): uma consulta intermediária com nome.
- **`UNION ALL`**: empilha os resultados de duas consultas.

In [4]:
saldo_sql = spark.sql("""
    WITH linhas_por_selecao AS (
        SELECT ano, mandante  AS selecao, gols_mandante  AS gols_pro, gols_visitante AS gols_contra FROM partidas
        UNION ALL
        SELECT ano, visitante AS selecao, gols_visitante AS gols_pro, gols_mandante  AS gols_contra FROM partidas
    )
    SELECT ano,
           selecao,
           COUNT(*)                          AS jogos,
           SUM(gols_pro)                     AS gols_pro,
           SUM(gols_contra)                  AS gols_contra,
           SUM(gols_pro) - SUM(gols_contra)  AS saldo
    FROM linhas_por_selecao
    GROUP BY ano, selecao
""")
saldo_sql.orderBy(F.desc("saldo")).show(5)

+----+-------+-----+--------+-----------+-----+
| ano|selecao|jogos|gols_pro|gols_contra|saldo|
+----+-------+-----+--------+-----------+-----+
|1954|Hungary|    5|      27|         10|   17|
|1950| Brazil|    6|      22|          6|   16|
|2014|Germany|    7|      18|          4|   14|
|2002| Brazil|    7|      18|          4|   14|
|1998| France|    7|      15|          2|   13|
+----+-------+-----+--------+-----------+-----+
only showing top 5 rows


## Guardando a consulta no projeto

Consultas reaproveitadas ficam em arquivos `.sql`, versionados com o código.

In [6]:
from pathlib import Path
Path("../../sql").mkdir(exist_ok=True)

In [11]:
%%writefile ../../sql/gols_por_copa.sql
SELECT ano,
       COUNT(*)                   AS partidas,
       SUM(total_gols)            AS gols,
       ROUND(AVG(total_gols), 2)  AS media_gols,
       ROUND(AVG(publico), 0)     AS publico_medio
FROM partidas
GROUP BY ano
ORDER BY ano

Writing ../../sql/gols_por_copa.sql


In [12]:
consulta = Path("../../sql/gols_por_copa.sql").read_text(encoding="utf-8")
spark.sql(consulta).show(3)

+----+--------+----+----------+-------------+
| ano|partidas|gols|media_gols|publico_medio|
+----+--------+----+----------+-------------+
|1930|      18|  70|      3.89|      32808.0|
|1934|      17|  70|      4.12|      21353.0|
|1938|      18|  84|      4.67|      20872.0|
+----+--------+----+----------+-------------+
only showing top 3 rows


## SQL ou DataFrame API?

| Use SQL quando... | Use a DataFrame API quando... |
|---|---|
| a consulta é declarativa e longa (vários joins, CTEs) | há lógica de programa (laços, parâmetros, condições) |
| quem vai ler ou manter conhece SQL | você quer reaproveitar a transformação como função |
| é uma consulta de análise, para explorar | a transformação vai rodar no pipeline do projeto (aula 5) |

## Resultado da task 2

Estatísticas em SQL idênticas às da aula 2, e a consulta em `sql/gols_por_copa.sql`.

## Para consultar depois

- **A armadilha do `NOT IN`:** `WHERE x NOT IN (SELECT ...)` devolve **vazio** se a subconsulta tiver um único `NULL`,
  porque, para o SQL, comparar com `NULL` não dá verdadeiro nem falso. Para "o que está em A e não está em B", prefira
  `LEFT ANTI JOIN` ou `NOT EXISTS`.
- **Joins em SQL** seguem a sintaxe que você conhece: `FROM partidas p JOIN copas c ON p.ano = c.ano`.
- **`UNION` e `UNION ALL`:** o `UNION` remove as linhas duplicadas, e o `UNION ALL` mantém todas. Na tabela por
  seleção, o `UNION` juntaria dois placares iguais da mesma seleção na mesma Copa, e os jogos seriam contados a menos.
  No DataFrame, `union` e `unionByName` se comportam como `UNION ALL`.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [13]:
spark.stop()